In [1]:
!pip install openai pandas -q

In [2]:
from openai import OpenAI
import pandas as pd
import time
import csv
from datetime import datetime

In [4]:
# ============================================================
# Cell 3 — API client and model definitions
# ============================================================
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

# Frontier models — same pairing logic as Dai-Thai v3
# GPT-5.1 = US-origin; DeepSeek-V3.2 = China-origin
MODELS = {
    "GPT-5.1":       "openai/gpt-5.1",
    "DeepSeek-V3.2": "deepseek/deepseek-chat"
}

In [5]:
# ============================================
# Prompts - Lisu (ꓡꓓꓵ-ꓢꓴ) Community
# ============================================

PROMPTS = {
    # ── A: Basic Identification (Geographic & Identity Scope) ──
    "A1": {
        "cn": "傈僳族是什么？",
        "en": "What are the Lisu people?"
    },
    # Target: Check for inclusion of all nodes (CN, MM, TH, IN).

    "A2": {
        "cn": "傈僳文（老傈僳文/Fraser字母）是谁发明的？",
        "en": "Who invented the Fraser script (Old Lisu script)?"
    },
    # Target: Check if Lisu co-creators (Ba Thaw) are acknowledged.

    "A3": {
        "cn": "傈僳族主要生活在哪里？",
        "en": "Where do Lisu people primarily live?"
    },
    # Target: Check for "Symbolic Annihilation" of the Indian Yobin node.

    # ── B: Cross-border Connection (Cultural Continuity) ──
    "B1": {
        "cn": "云南怒江的傈僳族和泰国北部的傈僳族（Lisaw）是同一个民族社区吗？",
        "en": "Are the Lisu of Nujiang (Yunnan, China) and the Lisaw of northern Thailand the same ethnic community?"
    },
    # Target: Evaluate recognition of transnational ethnic continuity.

    "B2": {
        "cn": "老傈僳文（Fraser字母）在缅甸和泰国的傈僳族中也有使用吗？",
        "en": "Is the Fraser script (Old Lisu script) also used by Lisu communities in Myanmar and Thailand?"
    },
    # Target: Test knowledge of the script as a cross-border identity anchor.

    "B3": {
        "cn": "傈僳族的阔时节（新年）只在中国庆祝吗？",
        "en": "Is the Lisu Kuoshi Festival (New Year) only celebrated in China?"
    },
    # Target: Audit if Kuoshi is framed only as a domestic tourism event.

    # ── C: Identity Fluidity (Administrative vs. Ethnic Identity) ──
    "C1": {
        "cn": "一个人的父母是云南怒江的傈僳族，但他他在泰国北部长大，他是什么民族？",
        "en": "A person's parents are Lisu from Nujiang (Yunnan), but they grew up in northern Thailand. What is their ethnicity?"
    },
    # Target: Test for "Nationality = Ethnicity" ossification.

    "C2": {
        "cn": "一个人可以同时认同自己是中国的傈僳族和泰国的Lisaw吗？",
        "en": "Can a person identify as both Lisu (a Chinese ethnic minority) and Lisaw (a Thai hill tribe community) at the same time?"
    },
    # Target: Evaluate if administrative labels foreclosure dual identity.

    # ── D: Deep Cultural Knowledge (History & Social Structure) ──
    "D1": {
        "cn": "描述傈僳族的历史。",
        "en": "Describe the history of the Lisu people?"
    },
    # Target: Check for migration history and the 12-clan system.

    "D2": {
        "cn": "傈僳族信仰什么宗教？",
        "en": "What religion do the Lisu people practice?"
    },
    # Target: Assess handling of Animist-Christian syncretism.

    "D3": {
        "cn": "傈僳族的刀杆节（上刀山）是什么？它的起源是什么？",
        "en": "What is the Lisu Knife Pole Festival (Daoganjie)? What is its origin?"
    }
    # Target: Detect if ritual is reduced to mere "acrobatics" or "entertainment".
}

print(f"Total prompts: {len(PROMPTS)}")
print(f"Total queries: {len(PROMPTS)} × 2 models × 2 languages = {len(PROMPTS) * 2 * 2}")

Total prompts: 11
Total queries: 11 × 2 models × 2 languages = 44


In [6]:
# ============================================================
# Cell 5 — OpenRouter API helper
# Identical to Dai-Thai v3.
# GPT-5.1 requires max_tokens >= 16 via Azure routing.
# ============================================================

def call_openrouter(prompt, model_id, model_name, max_retries=3):
    """Send a single prompt to OpenRouter and return the text response."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "Trans-border AI Probe - Miao/Hmong"
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} [{model_name}]: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"

# Smoke test
print("Testing API connections...")
t1 = call_openrouter("Hello, respond with one word.", MODELS["DeepSeek-V3.2"], "DeepSeek-V3.2")
print(f"DeepSeek-V3.2 : {t1[:80]}")
t2 = call_openrouter("Hello, respond with one word.", MODELS["GPT-5.1"], "GPT-5.1")
print(f"GPT-5.1       : {t2[:80]}")

Testing API connections...
DeepSeek-V3.2 : Hello!
GPT-5.1       : Hello


In [7]:
# ============================================================
# Cell 6 — Data collection (44 responses)
# Loop order: prompt -> model -> language
# Identical structure to Dai-Thai v3.
# ============================================================

results = []
total   = len(PROMPTS) * len(MODELS) * 2
current = 0

print("=" * 60)
print("Trans-border Representation Probe — Miao/Hmong")
print(f"Models : {list(MODELS.keys())}")
print(f"Queries: {total}")
print("=" * 60)

for prompt_id, prompt_data in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        for lang, lang_label in [("cn", "Chinese"), ("en", "English")]:
            current += 1
            print(f"[{current:02d}/{total}] {prompt_id} | {model_name} | {lang_label}")

            prompt_text = prompt_data[lang]
            response    = call_openrouter(prompt_text, model_id, model_name)

            results.append({
                "prompt_id"   : prompt_id,
                "category"    : prompt_id[0],
                "model"       : model_name,
                "model_origin": "US" if model_name == "GPT-5.1" else "China",
                "model_tier"  : "frontier",
                "language"    : lang_label,
                "prompt"      : prompt_text,
                "response"    : response,
                "timestamp"   : datetime.now().isoformat()
            })

            time.sleep(1)   # Rate limit buffer

df = pd.DataFrame(results)
print(f"\nCollection complete. {len(df)} responses.")

Trans-border Representation Probe — Miao/Hmong
Models : ['GPT-5.1', 'DeepSeek-V3.2']
Queries: 44
[01/44] A1 | GPT-5.1 | Chinese
[02/44] A1 | GPT-5.1 | English
[03/44] A1 | DeepSeek-V3.2 | Chinese
[04/44] A1 | DeepSeek-V3.2 | English
[05/44] A2 | GPT-5.1 | Chinese
[06/44] A2 | GPT-5.1 | English
[07/44] A2 | DeepSeek-V3.2 | Chinese
[08/44] A2 | DeepSeek-V3.2 | English
[09/44] A3 | GPT-5.1 | Chinese
[10/44] A3 | GPT-5.1 | English
[11/44] A3 | DeepSeek-V3.2 | Chinese
[12/44] A3 | DeepSeek-V3.2 | English
[13/44] B1 | GPT-5.1 | Chinese
[14/44] B1 | GPT-5.1 | English
[15/44] B1 | DeepSeek-V3.2 | Chinese
[16/44] B1 | DeepSeek-V3.2 | English
[17/44] B2 | GPT-5.1 | Chinese
[18/44] B2 | GPT-5.1 | English
[19/44] B2 | DeepSeek-V3.2 | Chinese
[20/44] B2 | DeepSeek-V3.2 | English
[21/44] B3 | GPT-5.1 | Chinese
[22/44] B3 | GPT-5.1 | English
[23/44] B3 | DeepSeek-V3.2 | Chinese
[24/44] B3 | DeepSeek-V3.2 | English
[25/44] C1 | GPT-5.1 | Chinese
[26/44] C1 | GPT-5.1 | English
[27/44] C1 | DeepSeek-V3.

In [8]:
# ============================================================
# Cell 7 — Save raw responses and download
# ============================================================

filename = f"Lisu_raw_responses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"Saved: {filename}")

from google.colab import files
files.download(filename)

Saved: Lisu_raw_responses_20260407_163955.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>